# Brecha Digital Territorial en Cusco

Cruce de luces nocturnas (NASA VNL 2025) con cobertura móvil OSIPTEL 2019 para identificar zonas con brecha digital activa en la región Cusco.

Pipeline: carga raster → reproyección a EPSG:4326 → normalización p2-p98 → índices (IBD, EDT) → mapas temáticos → clasificación 2x2 → stats.

## Step 0 — Setup

Imports y versiones de las librerías para reproducibilidad.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import seaborn as sns
import rasterio
from rasterio.warp import reproject, Resampling, calculate_default_transform
from rasterio.transform import array_bounds
import scipy
from scipy import ndimage, stats

print('rasterio  ', rasterio.__version__)
print('numpy     ', np.__version__)
print('matplotlib', plt.matplotlib.__version__)
print('scipy     ', scipy.__version__)
print('seaborn   ', sns.__version__)
print('pandas    ', pd.__version__)

sns.set_context('notebook')

# Rutas relativas. Soporta ejecución desde notebooks/ y desde la raiz del repo.
for candidate in [Path('../data'), Path('data')]:
    if candidate.exists():
        DATA = candidate.resolve()
        break
OUT = (DATA.parent / 'output')
OUT.mkdir(exist_ok=True)
print('DATA dir  ', DATA)
print('OUTPUT dir', OUT)

## Step 1 — Loading e inspección

Para cada raster imprimo CRS, shape, bandas, NoData, dtype, bbox, resolución y rango de valores.

In [ ]:
def km_per_pixel(transform, height, width, crs):
    """Aprox de tamaño de pixel en km, dependiendo del CRS."""
    dx, dy = abs(transform.a), abs(transform.e)
    if crs and crs.is_geographic:
        # En grados; convertir usando latitud central
        bounds = array_bounds(height, width, transform)
        lat_mid = 0.5 * (bounds[1] + bounds[3])
        kx = dx * 111.32 * np.cos(np.deg2rad(lat_mid))
        ky = dy * 110.57
        return dx, dy, kx, ky, 'deg'
    return dx, dy, dx / 1000.0, dy / 1000.0, 'm'


def inspect_raster(path, label):
    with rasterio.open(path) as ds:
        arr = ds.read(1, masked=True)
        dx, dy, kx, ky, unit = km_per_pixel(ds.transform, ds.height, ds.width, ds.crs)
        valid = int(arr.count())
        total = int(arr.size)
        amin = float(arr.min()) if valid else float('nan')
        amax = float(arr.max()) if valid else float('nan')
        print(f'=== {label} ===')
        print(f'  path     : {path.name}')
        print(f'  CRS      : {ds.crs}')
        print(f'  shape    : {ds.height} x {ds.width} (alto x ancho)')
        print(f'  bandas   : {ds.count}')
        print(f'  NoData   : {ds.nodata}')
        print(f'  dtype    : {ds.dtypes[0]}')
        print(f'  bbox     : {ds.bounds}')
        print(f'  pixel    : {dx:.6f} x {dy:.6f} {unit}  (~{kx*1000:.0f} x {ky*1000:.0f} m)')
        print(f'  validos  : {valid:,} / {total:,} pixeles')
        print(f'  rango    : [{amin:.4f}, {amax:.4f}]')
        print()
        return arr


VNL_PATH = DATA / 'VNL_cusco_2025.tif'
KERN_PATH = DATA / 'kernel_cobmovil2019_50m.tif'

vnl_arr = inspect_raster(VNL_PATH, 'VNL_cusco_2025 (NASA Black Marble)')
kern_arr = inspect_raster(KERN_PATH, 'kernel_cobmovil2019_50m (OSIPTEL)')